## Foundry Agent Tracing with Application Insights

### Installing Libraries and SDKs

In [ ]:
%pip install azure-ai-projects==2.3.0 opentelemetry-sdk azure-core-tracing-opentelemetry azure-monitor-opentelemetry opentelemetry-exporter-otlp-proto-http

### Importing Libraries and Utilities

In [1]:
import os
from azure.ai.projects.telemetry import AIProjectInstrumentor
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import trace

os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
# optional if you want prompt/response/tool args captured
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"


# configure exporter (you can pass conn string here once you have it)
# If you only know it after creating project_client, do it in next cell.

AIProjectInstrumentor().instrument()


### Setting up the Environment Variables

In [2]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

print(foundry_project_endpoint)
print(model_deployment_name)
print(mcp_server_name)

https://udemo-103-resource.services.ai.azure.com/api/projects/udemo-103
gpt-5.2
MSLearnMCP


In [3]:
from azure.ai.projects.telemetry import AIProjectInstrumentor
AIProjectInstrumentor().instrument()

### Creating the Foundry Project Client

In [4]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Instantiating an OpenAI Client

In [5]:
openai_client = project_client.get_openai_client()

### Fetching the MCP Server Connection Reference

In [6]:
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

The MCP Server Connection ID is: /subscriptions/0ad7156d-4b06-4b69-b6f3-4b5d990dd024/resourceGroups/udemo-103/providers/Microsoft.CognitiveServices/accounts/udemo-103-resource/projects/udemo-103/connections/MSLearnMCP


### Fetching the App Insights Connection String

In [7]:
connection_string = project_client.telemetry.get_application_insights_connection_string()
configure_azure_monitor(connection_string=connection_string)

print("App Insights Connection String: {}".format(connection_string))

App Insights Connection String: InstrumentationKey=53ca76f7-189f-4110-bb70-dca54f892335;IngestionEndpoint=https://swedencentral-0.in.applicationinsights.azure.com/;LiveEndpoint=https://swedencentral.livediagnostics.monitor.azure.com/;ApplicationId=1171d554-9818-4116-b7f6-8d576916a5b9


### Beginning the Trace

In [8]:
tracer = trace.get_tracer(__name__)

with tracer.start_as_current_span("agent-tracing-scenario"):

    tool = MCPTool(
        server_label = "microsoft_learn_server",
        server_url="https://learn.microsoft.com/api/mcp",
        require_approval="never",
        project_connection_id=connection_id
    )

    agent = project_client.agents.create_version(
        agent_name="MCP-Agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
            tools=[tool],
        )
    )

    print(f"Created MCP Agent with ID: {agent.id}")

    # create a conversation to use with the agent
    conversation = openai_client.conversations.create()
    print(f"Created conversation with id: {conversation.id}")

    user_query = "Can you pls help me with the latest ai foundry SDK code samples?" 

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": "MCP-Agent",
                "type": "agent_reference"
            }
        },
        input=user_query
    )

    print(f"Agent Response: {response.output_text}")

Created MCP Agent with ID: MCP-Agent:1
Created conversation with id: conv_0fd7dd2b13b6461d00XNNh3fuhBX6nVuEt5QReF9SpZniUBPGW
Agent Response: Latest **Microsoft Foundry (Azure AI Foundry) SDK** code samples are published in this quickstart (and kept current with the SDKs):

- **Quickstart: Get started with Microsoft Foundry SDK** (Python, C#, TypeScript, Java, REST)  
  https://learn.microsoft.com/azure/foundry/quickstarts/get-started-code

It includes ready-to-copy samples for:
- **Chat with a model** (Responses API)
- **Create an agent** (prompt agent definition)
- **Chat with an agent** (multi-turn using conversations)

### Key “latest” SDK packages referenced by the quickstart
- **Python**: `pip install "azure-ai-projects>=2.3.0" azure-identity`
- **C#**: `Azure.AI.Projects`, `Azure.AI.Projects.Agents`, `Azure.AI.Extensions.OpenAI`, `Azure.Identity`
- **TypeScript**: `npm install @azure/ai-projects @azure/identity`
- **Java**: `com.azure:azure-ai-agents` (plus `azure-identity`)

###